### Подготовка нужных функций  
Примечание: служебная часть, здесь происходит импорт библиотек и подготовка функции для формировании посылок

In [ ]:
import pandas as pd
import numpy as np

### Чтение данных  
Примечание: здесь происходит чтение и предобработка исходной таблицы. Исходная таблица должна содержать значения ***фактических*** показателей, на основе которых будет строиться прогноз.
Прошу обратить внимание на формат, в котором оказывается таблица перед отправкой на сервер: 
добавлена колонка с датой (ds), составленная из двух колонок - месяц и год.

In [ ]:
# Чтение исходного файла
path_to_file = 'Факт.xlsx'
df = pd.read_excel(path_to_file,sheet_name="Лист1",  header = 0)
df_plan = pd.read_excel('План.xlsx', sheet_name = 'План', header = 0)

df.rename(columns={'Календарный год': 'Год',
                   'Календарный месяц': 'Месяц',
                   '     Драйвер \n(загруженный)': 'Драйвер_шт'
                       }, inplace=True)
df_plan.rename(columns={'Календарный год': 'Год',
                        'Календарный месяц': 'Месяц',
                        '     Драйвер \n(загруженный)': 'Драйвер_шт_plan'
                       }, inplace=True)
for col in df.columns:
    print(repr(col)) 

In [ ]:
df_plan

### Обработка данных перед формированием поссылки (ч.1)
- ws_year_rules ***<-*** для каждой РО необходимо установить год из которого тянем данные, т.к в одной РО находится два года, что может задваивать данные
- Задаём тип данных в колонках для дальнейшей обработки
- df['ds'] ***<-*** формируем колонку с датой

In [ ]:
# Правила выбора года и методологии расчёта
ws_rules = {
    '24 в мет 24': [2023],  
    '25_в мет 25': [2024],
    '26_в мет 26': [2025,2026]
}

condition = pd.Series(False, index=df.index)

for area, years in ws_rules.items():
    condition |= (df['Расчёт'] == area) & (df['Год'].isin(years))

df_1 = df[condition]

df_2024 = df_plan[df_plan['Год'] == 2025]
df_2024['Год'] = 2024

df_2025 = df_plan[df_plan['Год'] == 2026]
df_2025['Год'] = 2025

df_plan = pd.concat([df_2024, df_2025], ignore_index = True)

In [ ]:
# Обработка даты
df_1['ds'] = df_1['Год'].astype('str') + '-' + df_1['Месяц'].astype('str') # Объединяем год и месяц в одной колонке
df_1['ds'] = pd.to_datetime(df_1.ds, format="%Y-%m") # Переводим в формат даты

df_plan['ds'] = df_plan['Год'].astype('str') + '-' + df_plan['Месяц'].astype('str') # Объединяем год и месяц в одной колонке
df_plan['ds'] = pd.to_datetime(df_plan.ds, format="%Y-%m") # Переводим в формат даты

### Обработка данных перед формированием поссылки (ч.2)
Здесь происходит подготовка таблиц к отправке на сервер.  
- Выставляем ограничения на необходимое количетсво наблюдей в данных, выставляем при необходимости ограничение по датам до которых забирается факт
- Объединяем данные из 2-х датафреймов
- lex(x)   **<-** общее количетсво наблюдений
- len(x[x['05. Календарный год']])   **<-**   количество наблюдений в отчётном периоде
- service_2026   **<-**   датафрейм из уникальных ЦФО+драйвер+услуга, которые сущесвуют в отчётном периоде

In [ ]:
#Для услуг которые существуют только в рамках 26 г, нужно создать фиктивный факт по плановым данным 26года
df_2025.rename(columns = {'Драйвер_шт_plan' : 'Драйвер_шт'}, inplace = True)
merged_actual = df_2025.merge(
    df_1[['Поставщик', 'Драйвер','Услуга','Год','Месяц']],
    on=['Поставщик', 'Драйвер','Услуга','Год','Месяц'],
    how='left',
    indicator=True
)
df_to_add = merged_actual[merged_actual['_merge'] == 'left_only'].drop(columns=['_merge']).copy()


df_final = pd.concat([df_1, df_to_add], ignore_index=True)

In [ ]:
# Обработка даты
df_final['ds'] = df_final['Год'].astype('str') + '-' + df_final['Месяц'].astype('str') # Объединяем год и месяц в одной колонке
df_final['ds'] = pd.to_datetime(df_final.ds, format="%Y-%m") # Переводим в формат даты

df_plan['ds'] = df_plan['Год'].astype('str') + '-' + df_plan['Месяц'].astype('str') # Объединяем год и месяц в одной колонке
df_plan['ds'] = pd.to_datetime(df_plan.ds, format="%Y-%m") # Переводим в формат даты

In [ ]:
# Создаем копию датафрейма
df_2 = df_final.copy()
# Создаем колонку с группировкой, по которой хотим строить прогноз
df_2['group'] = df_2['Поставщик'].astype(str) + '$' + df_2['Драйвер'].astype(str) + '$' + df_2['Услуга'].astype(str)
df_plan['group'] = df_plan['Поставщик'].astype(str) + '$' + df_plan['Драйвер'].astype(str) + '$' + df_plan['Услуга'].astype(str)
# Цепляем плановые данные в итоговый датафрейм
df_2 = df_2.merge(df_plan[['group', 'ds', 'Драйвер_шт_plan']], on=['group','ds'], how='left')
# Услуги+Драйвер+Поставщик, которые существуют в 2026г
service_2026 = df_2[df_2['Год'] == 2026]['group'].unique()


### Фильтры которые в теории можно выключить
1) Фильтр только на услуги факта текущего года
- df_2= df_2[df_2['group'].isin(service_2026)]

2) Фильтр и проверка достаточности данных

- year = 2026 # <- Текущий год

- obs_in_curr_year = 1  <- Сколько месяцов минимум должна существовать услуга в текущем году

- obs_all_periods = 1  <- Сколько месяцев минимум услуга должна существовать за все годы

- df_2 = df_2.groupby('group').filter(
    lambda x: (len(x[x['Год'] == year]) >= obs_in_curr_year) and (len(x) >= obs_all_periods)
)

In [ ]:
year = 2026 # <- Текущий год
obs_in_curr_year = 1 # Сколько месяцов минимум должна существовать услуга в текущем году
obs_all_periods = 36 # Сколько месяцев минимум услуга должна существовать за все годы
df_2 = df_2.groupby('group').filter( lambda x: (len(x[x['Год'] == year]) >= obs_in_curr_year) and (len(x) >= obs_all_periods) )

### Обработка данных перед формированием поссылки (ч.3)
- df['Темп роста_%'] **<-** Расчёт темпов роста помесячно для факт предыдущего годов и плана отчёнотного периода
- df['CV'] **<-** Расчёт коэффициента вариации
- df['Upper|Lower_bound'] **<-** Расчёт верхней/нижней границы

In [ ]:
#Ограничение до какой даты забрать данные в расчёт
df_2 = df_2[(df_2['ds'] < '2026-04-01')]# & df_2['ds'] > '2024-01-01')]
# Оставляем только нужные столбцы
df_2 = df_2[['ds', 'Год', 'group', 'Услуга', 'Драйвер_шт', 'Драйвер_шт_plan']]
# Считаем статистики для факта
df_2['Темп_роста_%'] = (df_2.groupby('group')['Драйвер_шт'].pct_change()).round(2)
df_2['CV'] = df_2.groupby(['group', 'Год'])['Драйвер_шт'].transform(lambda x: x.std() / x.mean()).round(2)
df_2['Upper_bound'] = 1 + df_2['CV'].round(2)
df_2['Lower_bound'] = 1 - df_2['CV'].round(2)
df_2['Lower_bound'] = df_2['Lower_bound'].clip(lower=0).round(2)
df_2['Темп_роста_%'] = df_2['Темп_роста_%'].fillna(0).round(2)
# Считаем статистики для плана
df_2['Темп_роста_%_plan'] = df_2.groupby('group')['Драйвер_шт_plan'].pct_change().round(2)
df_2['CV_plan'] = df_2.groupby(['group', 'Год'])['Драйвер_шт_plan'].transform(lambda x: x.std() / x.mean()).round(2)
df_2['Upper_bound_plan'] = 1 + df_2['CV_plan'].round(2)
df_2['Lower_bound_plan'] = 1 - df_2['CV_plan'].round(2)
df_2['Lower_bound_plan'] = df_2['Lower_bound_plan'].clip(lower=0).round(2)
df_2['Темп_роста_%_plan'] = df_2['Темп_роста_%_plan'].fillna(0).round(2)

In [ ]:
# Условие для поиска отклоенений 
condition = (
    ((df_2['Темп_роста_%'] < df_2['Lower_bound']) | (df_2['Темп_роста_%'] > df_2['Upper_bound'])) &
    (df_2['Темп_роста_%_plan'] >= df_2['Lower_bound_plan']) & (df_2['Темп_роста_%_plan'] <= df_2['Upper_bound_plan'])
)

# Среднее за год 
df_2['AVG'] = df_2.groupby(['group', 'Год'])['Драйвер_шт'].transform('mean').round(2)

# Замена
df_2.loc[condition, 'Драйвер_шт'] = df_2.loc[condition, 'AVG']


### Формирование посылки    
- horizon = 12    **<-**   горизонт планирования (месяцев)
- groupby_column='Драйвер+ГОСБ'   **<-**   название колонки с группировкой, по которой хотим строить прогноз
- time_column='ds'   **<-** название колонки с датой
- target_column='Обороты'   **<-**   название колонки со значениями, на которых будет строиться прогноз
- collapse_clusters = True   **<-**   если True, считает быстрее, но менее точно, если False, то медленнее (в 3-4 раза), но точнее
- future = True   **<-**   Предсказываем прошлое или будущее (future = True - будущее)

In [ ]:
# Оставляем только нужные столбцы
df_package = df_2[['ds', 'group', 'Драйвер_шт']]
# Переименовываем колонку с данными, на которых сроим прогноз, в target
df_package.rename(columns={'Драйвер_шт': 'target'}, inplace=True)

In [ ]:
# Приводим колонку с датой к строковому типу
df_package['ds'] = df_2.ds.astype('str')
# Проверить результат
df_package.head(5)

In [ ]:
df_package.groupby("group").count().value_counts()

In [ ]:
df_2_gr = pd.DataFrame(
    df_package.pivot_table(
        index='group',
        columns='ds',
        values='target',
        aggfunc='sum',
        fill_value=0
    )
    .stack()  # Преобразуем в MultiIndex Series
    .sort_index(level=['group', 'ds'])
    .rename('target')  # Если нужно сохранить имя столбца
)
df_2_gr

In [ ]:
print(
    df_2[
        df_2['ds'].dt.year == 2025
    ][['ds', 'group', 'target']].to_string(index=False)
)